In [1]:
import json
from datetime import datetime, timezone, timedelta

import boto3
from botocore.exceptions import ClientError, BotoCoreError

In [2]:
REGION = "us-east-2"

sqs = boto3.client("sqs", region_name=REGION)
cloudtrail = boto3.client("cloudtrail", region_name=REGION)
iam = boto3.client("iam", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

QUEUE_URL = "https://sqs.us-east-2.amazonaws.com/585192672941/decoy-events-aggregated"

In [3]:
def iso_to_dt(value: str) -> datetime:
    if not value:
        raise ValueError("Datetime string cannot be empty")

    value = value.strip()
    if value.endswith("Z"):
        value = value.replace("Z", "+00:00")

    dt = datetime.fromisoformat(value)

    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt

In [4]:
def read_one_message(queue_url: str, wait_time_seconds: int = 10, visibility_timeout: int = 60):
    response = sqs.receive_message(
        QueueUrl=queue_url,
        MaxNumberOfMessages=1,
        WaitTimeSeconds=wait_time_seconds,
        VisibilityTimeout=visibility_timeout,
        AttributeNames=["All"],
        MessageAttributeNames=["All"],
    )

    messages = response.get("Messages", [])

    if not messages:
        return None

    msg = messages[0]

    try:
        body = json.loads(msg["Body"])
    except Exception:
        body = msg["Body"]

    return {
        "message_id": msg.get("MessageId"),
        "receipt_handle": msg.get("ReceiptHandle"),
        "body": body,
        "attributes": msg.get("Attributes", {}),
        "message_attributes": msg.get("MessageAttributes", {}),
    }

In [5]:
def delete_sqs_message(queue_url: str, receipt_handle: str):
    sqs.delete_message(
        QueueUrl=queue_url,
        ReceiptHandle=receipt_handle,
    )
    return {"deleted": True}

In [6]:
def query_cloudtrail(
    source_ip: str = None,
    access_key_id: str = None,
    username: str = None,
    start_time: str = "",
    end_time: str = "",
    max_results: int = 20,
):
    if not start_time or not end_time:
        raise ValueError("start_time and end_time are required")

    if not any([source_ip, access_key_id, username]):
        raise ValueError("At least one of source_ip, access_key_id, or username must be provided")

    start_dt = iso_to_dt(start_time)
    end_dt = iso_to_dt(end_time)

    lookup_attrs = []
    if username:
        lookup_attrs.append({"AttributeKey": "Username", "AttributeValue": username})
    if access_key_id:
        lookup_attrs.append({"AttributeKey": "AccessKeyId", "AttributeValue": access_key_id})

    events = []
    seen_event_ids = set()

    try:
        if lookup_attrs:
            for attr in lookup_attrs:
                next_token = None

                while True:
                    params = {
                        "LookupAttributes": [attr],
                        "StartTime": start_dt,
                        "EndTime": end_dt,
                        "MaxResults": min(max_results, 50),
                    }
                    if next_token:
                        params["NextToken"] = next_token

                    response = cloudtrail.lookup_events(**params)

                    for event in response.get("Events", []):
                        event_id = event.get("EventId")
                        if event_id not in seen_event_ids:
                            seen_event_ids.add(event_id)
                            events.append(event)

                    next_token = response.get("NextToken")
                    if not next_token or len(events) >= max_results:
                        break
        else:
            next_token = None

            while True:
                params = {
                    "StartTime": start_dt,
                    "EndTime": end_dt,
                    "MaxResults": min(max_results, 50),
                }
                if next_token:
                    params["NextToken"] = next_token

                response = cloudtrail.lookup_events(**params)

                for event in response.get("Events", []):
                    event_id = event.get("EventId")
                    if event_id not in seen_event_ids:
                        seen_event_ids.add(event_id)
                        events.append(event)

                next_token = response.get("NextToken")
                if not next_token or len(events) >= max_results:
                    break

        if source_ip:
            filtered = []
            for event in events:
                raw = event.get("CloudTrailEvent", "")
                if source_ip in raw:
                    filtered.append(event)
            events = filtered

        return {
            "source": "cloudtrail",
            "event_count": len(events[:max_results]),
            "events": events[:max_results],
        }

    except (ClientError, BotoCoreError) as e:
        return {
            "source": "cloudtrail",
            "event_count": 0,
            "events": [],
            "error": str(e),
        }

In [7]:
def query_identity_context(
    username: str = None,
    role_name: str = None,
    access_key_id: str = None,
):
    result = {
        "source": "identity_context",
        "user": None,
        "role": None,
        "access_key_matches": [],
    }

    if username:
        try:
            response = iam.get_user(UserName=username)
            result["user"] = response.get("User", {})
        except iam.exceptions.NoSuchEntityException:
            result["user"] = {"message": f"user '{username}' not found"}
        except (ClientError, BotoCoreError) as e:
            result["user"] = {"error": str(e)}

    if role_name:
        try:
            response = iam.get_role(RoleName=role_name)
            result["role"] = response.get("Role", {})
        except iam.exceptions.NoSuchEntityException:
            result["role"] = {"message": f"role '{role_name}' not found"}
        except (ClientError, BotoCoreError) as e:
            result["role"] = {"error": str(e)}

    if username and access_key_id:
        try:
            keys = iam.list_access_keys(UserName=username).get("AccessKeyMetadata", [])
            result["access_key_matches"] = [
                k for k in keys if k.get("AccessKeyId") == access_key_id
            ]
        except iam.exceptions.NoSuchEntityException:
            result["access_key_matches"] = []
        except (ClientError, BotoCoreError) as e:
            result["access_key_matches"] = [{"error": str(e)}]

    return result

In [8]:
def query_s3_security_context(bucket_name: str):
    result = {
        "source": "s3_security_context",
        "bucket_name": bucket_name,
        "bucket_policy_status": None,
        "public_access_block": None,
        "bucket_versioning": None,
    }

    if not bucket_name:
        result["error"] = "bucket_name not provided"
        return result

    try:
        policy_status = s3.get_bucket_policy_status(Bucket=bucket_name)
        result["bucket_policy_status"] = policy_status.get("PolicyStatus", {})
    except Exception as e:
        result["bucket_policy_status"] = {"error": str(e)}

    try:
        pab = s3.get_public_access_block(Bucket=bucket_name)
        result["public_access_block"] = pab.get("PublicAccessBlockConfiguration", {})
    except Exception as e:
        result["public_access_block"] = {"error": str(e)}

    try:
        versioning = s3.get_bucket_versioning(Bucket=bucket_name)
        result["bucket_versioning"] = versioning
    except Exception as e:
        result["bucket_versioning"] = {"error": str(e)}

    return result

In [9]:
def extract_pivots_from_message_body(body: dict):
    """
    Extract common investigation pivots from the SQS payload.
    Adjust field mapping here if your payload changes.
    """
    summary = body.get("summary", {}) if isinstance(body, dict) else {}

    source_ip = (
        summary.get("sourceIP")
        or summary.get("sourceIp")
        or body.get("sourceIP")
        or body.get("sourceIp")
    )

    access_key_id = (
        summary.get("accessKeyId")
        or body.get("accessKeyId")
    )

    username = (
        summary.get("userName")
        or summary.get("username")
        or body.get("userName")
        or body.get("username")
    )

    bucket_name = (
        summary.get("bucket")
        or summary.get("bucketName")
        or body.get("bucket")
        or body.get("bucketName")
    )

    role_name = (
        summary.get("roleName")
        or body.get("roleName")
    )

    start_time = body.get("firstSeen")
    end_time = body.get("lastSeen")

    return {
        "source_ip": source_ip,
        "access_key_id": access_key_id,
        "username": username,
        "role_name": role_name,
        "bucket_name": bucket_name,
        "start_time": start_time,
        "end_time": end_time,
    }

In [10]:
def expand_time_window(start_time: str, end_time: str, lookback_minutes: int = 15, forward_minutes: int = 15):
    start_dt = iso_to_dt(start_time) - timedelta(minutes=lookback_minutes)
    end_dt = iso_to_dt(end_time) + timedelta(minutes=forward_minutes)

    return {
        "start_time": start_dt.isoformat().replace("+00:00", "Z"),
        "end_time": end_dt.isoformat().replace("+00:00", "Z"),
    }

In [11]:
def summarize_cloudtrail_events(events):
    summarized = []

    for event in events:
        summarized.append({
            "eventTime": str(event.get("EventTime")),
            "eventName": event.get("EventName"),
            "username": event.get("Username"),
            "eventId": event.get("EventId"),
            "resources": event.get("Resources", []),
        })

    return summarized

In [42]:
def build_incident_report_prompt(cloudtrail_data, iam_data, s3_data, sqs_message_body):
    system_prompt = """
You are a senior cloud security analyst specializing in AWS incident response.

Your task is to analyze structured AWS security data and produce a clear, concise, and technically accurate incident report.

This is a Decoy Incident. nobody should access the S3 bucket. if there is access to the S3 bucket, it is a strong indicator of compromise and malicious activity.

Focus on:
- Reconstructing attacker behavior
- Identifying identity and access patterns
- Highlighting suspicious actions
- Explaining security impact
- Providing actionable recommendations

Do not hallucinate. Use only the provided data.
If data is missing, explicitly state assumptions.
""".strip()

    user_prompt = f"""
You are given structured AWS security investigation data from multiple sources.

Your task is to generate a detailed incident report.

========================
INPUT DATA
========================

1. SQS Session Events (Primary Alert Events):
These are the grouped session events that triggered the alert. They are the PRIMARY source for the timeline and attack-path reconstruction.
{json.dumps(sqs_message_body, indent=2, default=str)}

2. CloudTrail Events (Enrichment / Context):
These are additional CloudTrail log records fetched around the same time window. Use them to enrich and corroborate the SQS session events.
{json.dumps(cloudtrail_data, indent=2, default=str)}

3. IAM Identity Context:
{json.dumps(iam_data, indent=2, default=str)}

4. S3 Access Decision Context:
{json.dumps(s3_data, indent=2, default=str)}

========================
INSTRUCTIONS
========================

Analyze the data and generate a structured incident report with the following sections:

1. Executive Summary
- Brief description of what happened
- Key risk level (Low / Medium / High / Critical)

2. Timeline of Events
- Build the timeline primarily from the SQS Session Events (source 1). These are the events that triggered the alert.
- Merge in any additional CloudTrail events (source 2) that add context or fill gaps.
- For each entry include: timestamp, event name, action performed, and source (SQS Session / CloudTrail).
- Order all events chronologically regardless of source.
- Highlight suspicious or abnormal behavior

3. Identity Analysis
- Who performed the actions (user, role, assumed role, etc.)
- Access key usage (if present)
- Any anomalies in identity behavior

4. Attack Pattern / Behavior Analysis
- Describe attacker intent (enumeration, data access, exfiltration, etc.)
- Identify sequence patterns (for example: List -> Get -> Delete)
- Mention failed vs successful attempts

5. S3 Access & Exposure Analysis
- Was access allowed or denied?
- Was the bucket publicly accessible?
- Any misconfigurations or security risks

6. Impact Assessment
- What data or resources were affected?
- Potential business/security impact

7. Indicators of Compromise (IOCs)
- Source IP
- Access Key IDs
- Usernames / Role ARNs

8. Recommendations
- Immediate actions
- Preventive measures

IMPORTANT RULES:
- Do not invent missing data
- Clearly state 'Not Available' where applicable
- Base conclusions strictly on provided data
""".strip()

    return {
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
    }

In [13]:
def generate_report(openai_client, model: str, prompt_bundle: dict):
    response = openai_client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": prompt_bundle["system_prompt"]},
            {"role": "user", "content": prompt_bundle["user_prompt"]},
        ],
    )

    return response.output_text

In [43]:
def enrich_message(message: dict):
    body = message["body"]
    pivots = extract_pivots_from_message_body(body)

    source_ip = pivots["source_ip"]
    access_key_id = pivots["access_key_id"]
    username = pivots["username"]
    role_name = pivots["role_name"]
    bucket_name = pivots["bucket_name"]
    start_time = pivots["start_time"]
    end_time = pivots["end_time"]

    if start_time and end_time:
        expanded = expand_time_window(start_time, end_time, lookback_minutes=15, forward_minutes=2)
        ct_start_time = expanded["start_time"]
        ct_end_time = expanded["end_time"]
    else:
        ct_start_time = start_time
        ct_end_time = end_time

    cloudtrail_result = {}
    if ct_start_time and ct_end_time and any([source_ip, access_key_id, username]):
        cloudtrail_result = query_cloudtrail(
            source_ip=source_ip,
            access_key_id=access_key_id,
            username=username,
            start_time=ct_start_time,
            end_time=ct_end_time,
            max_results=20,
        )
    else:
        cloudtrail_result = {
            "source": "cloudtrail",
            "event_count": 0,
            "events": [],
            "error": "Insufficient pivots or missing time window",
        }

    identity_result = query_identity_context(
        username=username,
        role_name=role_name,
        access_key_id=access_key_id,
    )

    s3_result = query_s3_security_context(bucket_name=bucket_name)

    enriched = {
        "message_metadata": {
            "message_id": message.get("message_id"),
            "attributes": message.get("attributes", {}),
        },
        "raw_message_body": body.get("sessionEvents", []),
        "pivots": pivots,
        "cloudtrail": cloudtrail_result,
        "identity_context": identity_result,
        "s3_context": s3_result,
    }

    return enriched

In [44]:
def process_sqs_message(
    openai_client,
    model: str,
    queue_url: str,
    delete_after_success: bool = False,
):
    message = read_one_message(queue_url)

    if not message:
        return {
            "status": "no_message",
            "message": "No messages found in SQS queue."
        }

    enriched = enrich_message(message)

    cloudtrail_summary = summarize_cloudtrail_events(
        enriched["cloudtrail"].get("events", [])
    )

    prompt_bundle = build_incident_report_prompt(
        cloudtrail_data=cloudtrail_summary,
        iam_data=enriched["identity_context"],
        s3_data=enriched["s3_context"],
        sqs_message_body=enriched["raw_message_body"],
    )

    report_text = generate_report(
        openai_client=openai_client,
        model=model,
        prompt_bundle=prompt_bundle,
    )

    result = {
        "status": "processed",
        "message_id": message.get("message_id"),
        "receipt_handle": message.get("receipt_handle"),
        "enriched_data": enriched,
        "prompt_bundle": prompt_bundle,
        "incident_report": report_text,
    }

    if delete_after_success:
        delete_sqs_message(
            queue_url=queue_url,
            receipt_handle=message["receipt_handle"]
        )
        result["sqs_delete_status"] = "deleted"
    else:
        result["sqs_delete_status"] = "not_deleted"

    return result

In [16]:
# Example:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key or api_key == "your_openai_api_key_here":
    raise ValueError("Set a real OPENAI_API_KEY in .env before running this cell.")

print("OPENAI_API_KEY loaded successfully")

client = OpenAI(api_key=api_key)

OPENAI_API_KEY loaded successfully


In [45]:
result = process_sqs_message(
    openai_client=client,
    model="gpt-4o-mini",   # or your chosen model
    queue_url=QUEUE_URL,
    delete_after_success=False
)

In [37]:
print(result["status"])
print(result["sqs_delete_status"])
print(result["message_id"])

processed
not_deleted
df629a0b-b06e-4824-90a1-f2024efcdeb2


In [47]:
print(result["incident_report"])

# Incident Report

## 1. Executive Summary
On April 1, 2026, unauthorized access to the S3 bucket `corp-finance-archive-2023` was detected, performed by an IAM user named Mohan. The user engaged in various actions typical of an unauthorized intrusion including bucket enumeration and the retrieval of sensitive objects. Given that the S3 bucket was a decoy and should not have been accessed, this incident is categorized as **Critical** with a high risk of data exfiltration.

## 2. Timeline of Events
| Timestamp (UTC)           | Event Name                           | Action Performed                | Source       |
|---------------------------|--------------------------------------|----------------------------------|--------------|
| 2026-04-01T02:39:51Z      | DescribeRegions                      | Enumerate AWS regions            | CloudTrail   |
| 2026-04-01T02:39:52Z      | HeadBucket                           | Check bucket existence           | SQS Session  |
| 2026-04-01T02:39:56Z 

In [46]:
sqs_data = result.get("enriched_data", {}).get("raw_message_body")
print("type:", type(sqs_data).__name__)
print("prompt has SQS section:", "SQS Session Events" in result.get("prompt_bundle", {}).get("user_prompt", ""))
if isinstance(sqs_data, list):
    print("count:", len(sqs_data))
    if sqs_data:
        print("sample keys:", list(sqs_data[0].keys()) if isinstance(sqs_data[0], dict) else sqs_data[0])
elif isinstance(sqs_data, dict):
    print("keys:", list(sqs_data.keys())[:10])

type: list
prompt has SQS section: True
count: 4
sample keys: ['eventTime', 'eventName', 'eventSource', 'eventID', 'eventCategory', 'managementEvent', 'readOnly', 'userType', 'userName', 'principalId', 'arn', 'accessKeyId', 'sourceIPAddress', 'userAgent', 'awsRegion', 'requestID', 'recipientAccountId', 'bucketName', 'objectKey', 'resources', 'errorCode', 'errorMessage', 'bytesTransferredIn', 'bytesTransferredOut', 'authenticationMethod', 'signatureVersion', 'tlsVersion', 'cipherSuite']
